# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Authors: {meta.author}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the Croissant schema to enumerate available record sets (`@id`), their fields (`@id`), and columns.

In [ ]:
# List all record sets by `@id` and explore their fields
record_sets = list(dataset.record_sets())
print("Available Record Sets (by @id):")
for rset in record_sets:
    print(f"- RecordSet @id: {rset['@id']} (name: {rset.get('name', 'N/A')})")
    print("  Fields:")
    fields = rset.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            field_id = field.get('@id', str(field))
            field_name = field.get('name', 'N/A')
            print(f"    * Field @id: {field_id} (name: {field_name})")
        else:
            print(f"    * Field @id: {field}")
    print("")
# Optionally, display all field IDs for later referencing
all_recordset_ids = [rset['@id'] for rset in record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** For illustration, we extract data from the first available record set.

In [ ]:
# Extract data from each record set
dataframes = {}

# For demonstration, use the first available record set
for record_set_id in all_recordset_ids:
    print(f"Loading data for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print("Columns (@id):", df.columns.tolist())
        display(df.head())

# Select first loaded record set for further analysis
main_record_set_id = list(dataframes.keys())[0] if len(dataframes) > 0 else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field (e.g., `cr:peptideCount` if available) for demonstration, filter on it, normalize, and group by a categorical field (e.g., `cr:description`).

In [ ]:
# Choose representative fields, using @id for all references
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    print(f"Columns in main RecordSet ({main_record_set_id}):\n", df.columns.tolist())
    # Try to select a likely numeric field
    potential_numeric_ids = [col for col in df.columns if any(x in col.lower() for x in ["count", "mw", "abundance", "coverage", "peptide", "number", "score"])]
    if len(potential_numeric_ids) == 0:
        print("No numeric columns found -- please check available columns.")
    else:
        numeric_field = potential_numeric_ids[0]
        print(f"Using numeric field: {numeric_field}")
        # convert to numeric, errors='coerce' ensures non-parsable values become NaN
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].median() if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        if filtered_df[numeric_field].std() != 0:
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to pick a group field
        potential_group_field = [col for col in df.columns if col != numeric_field][0] if len(df.columns) > 1 else numeric_field
        if potential_group_field:
            grouped_df = filtered_df.groupby(potential_group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {potential_group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
else:
    print("No usable record set loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For demonstration, a histogram of the selected numeric field, and a bar plot of means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and len(dataframes) > 0 and 'filtered_df' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.show()
    # Grouped barplot if possible
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        plot_df = grouped_df.head(20)  # show only top 20 for clarity
        sns.barplot(x=potential_group_field, y=numeric_field, data=plot_df)
        plt.xticks(rotation=60, ha='right')
        plt.title(f"Mean {numeric_field} by {potential_group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR<sup>2</sup> mass spectrometry dataset from Croissant schema, explored the available record sets and fields via their `@id`, loaded data via `mlcroissant`, and performed basic EDA including filtering, normalization, grouping, and visualization.

All references to entities (record sets, fields) are via their Croissant `@id` throughout, ensuring reproducibility and schema-faithful exploration.

Further analysis can build upon this structure to answer domain-specific biological or data-processing questions relevant to extracellular vesicle protein analysis in human mast cells.